# MEIKA Vox · Santa María\nTranscripción + diarización de audios privados desde Google Drive.\n\n**Haz esto:** activa GPU en Colab y ejecuta todas las celdas. El audio nunca se sube al repo.\n

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg git\n!python -m pip install -q --upgrade pip\n!rm -rf /content/meika-vox\n!git clone -q https://github.com/MeikaLab/meika-vox.git\n%cd /content/meika-vox\n!python -m pip install -q -e '.[whisperx]'\n!meika-vox doctor\n

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n

## Diarización\nMEIKA Vox usa pyannote. Si guardaste `HF_TOKEN` en los secretos de Colab lo toma solo; si no, te lo pedirá oculto una vez.\n

In [ ]:
import os\nfrom getpass import getpass\n\ntry:\n    from google.colab import userdata\n    token = userdata.get('HF_TOKEN')\nexcept Exception:\n    token = None\n\nif not token:\n    token = getpass('Pega tu HF_TOKEN para diarización: ')\nos.environ['HF_TOKEN'] = token\nprint('HF_TOKEN cargado:', bool(token))\n!meika-vox doctor\n

## Audios\nCambia solo estas rutas si tus archivos están en otra carpeta.\n

In [ ]:
from pathlib import Path\n\nAUDIO_LAS_CABRAS = Path('/content/drive/MyDrive/José Contreras 2.m4a')\nAUDIO_ESCUELA = Path('/content/drive/MyDrive/Grabación mesa 1.m4a')\nOUTPUT = Path('/content/drive/MyDrive/MEIKA_Vox/Santa_Maria_2026')\nOUTPUT.mkdir(parents=True, exist_ok=True)\n\nAUDIOS = [AUDIO_LAS_CABRAS, AUDIO_ESCUELA]\nfor p in AUDIOS:\n    print(p, '✅' if p.exists() else '❌ NO ENCONTRADO')\n

## Procesar\nUsa `large-v3`, glosario Santa María, diarización y el guard anti-repetición de Vox.\n

In [ ]:
import subprocess\n\ndef transcribe(audio: Path):\n    cmd = [\n        'meika-vox','transcribe',str(audio),\n        '--project-id','SM26',\n        '--language','es',\n        '--model','large-v3',\n        '--diarize',\n        '--glossary','configs/glossaries/santa_maria_2026.json',\n        '--output',str(OUTPUT),\n    ]\n    print('Procesando:', audio.name)\n    subprocess.run(cmd, check=True)\n\nfor audio in AUDIOS:\n    transcribe(audio)\n

## Resultado\nBusca `transcript_normalized.txt` para leer, `speaker_turns.jsonl` para hablantes y `qa.json` para detectar problemas. `text_raw` nunca se modifica.\n

In [ ]:
runs = []\nfor manifest in OUTPUT.rglob('manifest.json'):\n    runs.append(manifest.parent)\n\nfor run in sorted(runs, key=lambda p: p.stat().st_mtime):\n    print('\nRUN:', run)\n    for name in [\n        'transcript_normalized.txt',\n        'speaker_turns.jsonl',\n        'normalization_changes.json',\n        'qa.json',\n    ]:\n        p = run / name\n        if p.exists(): print('  ✅', name)\n